# Reinforcement Learning: Q-Learning on Cliff Walking

**Watch an AI learn to walk across a cliff!** Starting from complete chaos (falling off immediately), the agent learns to find the optimal path by the end.

## The Problem:
- **Goal:** Walk from START (green) to GOAL (gold) 
- **Danger:** A CLIFF in the middle (red) = instant failure (-100 penalty)
- **Challenge:** Find the optimal path without falling!

## What You'll See:
- **Episode 1:** Agent stumbles around, falls off cliff immediately ❌
- **Episode 20:** Agent cautious, takes long safe path ⚠️
- **Episode 100:** Agent learning, faster paths ✅
- **Episode 300:** Agent getting risky, walks edge of cliff 🎯
- **Episode 499:** Agent expert, perfect speedrun on cliff edge ⚡

## Cell 1: The Setup & Imports

Bring in all the tools we need.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path
import time

np.random.seed(42)
print('✓ Libraries loaded!')

## Cell 2: The Brain & Hyperparameters

Set up the environment and Q-Table. Note: 48 states (4 rows × 12 cols) and 4 actions (up, down, left, right).

In [ ]:
# Initialize the environment
env = gym.make('CliffWalking-v0')

# Q-Table: 48 possible states × 4 possible actions
q_table = np.zeros((48, 4))

# Reinforcement Learning Hyperparameters
alpha = 0.1          # Learning rate: how much to update each step
gamma = 0.99         # Discount factor: value of future rewards
epsilon = 1.0        # Exploration rate: 100% random at start
epsilon_decay = 0.995
min_epsilon = 0.01   # Minimum exploration (always try new things)

print('✓ Environment initialized!')
print(f'  States: 48 (4 rows × 12 columns)')
print(f'  Actions: 4 (up, down, left, right)')
print(f'  Q-Table shape: {q_table.shape}')

## Cell 3: The Visualization Engine

Function to draw the cliff walking grid. Shows:
- **Green:** Start position
- **Red:** Cliff (danger!)
- **Gold:** Goal
- **Blue dot:** Current agent position

In [ ]:
def draw_grid(state, episode_num, title='Agent Position'):
    """Draw the 4x12 cliff walking grid with agent position"""
    fig, ax = plt.subplots(figsize=(14, 3))
    
    # Create grid: 4 rows, 12 columns
    grid = np.zeros((4, 12))
    
    # Mark the cliff (row 3, columns 1-10): value = -1 (red)
    grid[3, 1:11] = -1
    
    # Mark the goal (row 3, column 11): value = 2 (gold)
    grid[3, 11] = 2
    
    # Mark the agent: value = 1 (blue)
    # Convert state (0-47) to row, col
    agent_row = state // 12
    agent_col = state % 12
    grid[agent_row, agent_col] = 1
    
    # Display with color map
    # Blue for agent, Red for cliff, Yellow/Gold for goal
    cax = ax.matshow(grid, cmap='RdYlBu', vmin=-1, vmax=2)
    
    ax.set_title(f'Episode {episode_num}: {title}', fontsize=14, pad=15)
    
    # Draw grid lines
    ax.set_xticks(np.arange(-.5, 12, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
    ax.grid(which='minor', color='black', linestyle='-', linewidth=1)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Add legend
    ax.text(-0.8, -0.8, '🟢 START', fontsize=10)
    ax.text(4, -0.8, '🔴 CLIFF', fontsize=10)
    ax.text(10, -0.8, '🏆 GOAL', fontsize=10)
    
    plt.tight_layout()
    return fig

print('✓ Visualization engine ready!')

## Cell 4: Run One Episode to Show the Concept

Before training, let's test the visualization:

In [ ]:
# Run one random episode to show visualization
state, _ = env.reset()
fig = draw_grid(state, episode_num=0, title='Starting Position (before training)')
plt.show()

print('✓ You just saw what the grid looks like!')
print('  Green = Start position')
print('  Red = Cliff (fall = -100 penalty!)')
print('  Gold = Goal position')
print('  Blue = Current agent position')

## Cell 5: The Training Loop with Checkpoints

Train silently in the background, but pause at specific episodes to show progress. We'll capture frames to make GIFs!

In [ ]:
total_episodes = 500
checkpoints = [1, 20, 100, 300, 499]  # Episodes where we pause to visualize
episode_rewards = []  # Track reward each episode
checkpoint_data = {}  # Store agent paths at checkpoints

print('\n' + '='*60)
print('🧠 TRAINING THE AGENT (500 EPISODES)')
print('='*60)

for episode in range(total_episodes):
    state, _ = env.reset()
    episode_reward = 0
    done = False
    path = [state]  # Track path for GIF generation
    
    # Training loop for this episode
    while not done:
        # Epsilon-Greedy Strategy
        if np.random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()  # Explore: random action
        else:
            action = np.argmax(q_table[state])  # Exploit: best known action
        
        # Take action
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Q-Learning Update (The Brain Learning!)
        old_q = q_table[state, action]
        best_next_q = np.max(q_table[next_state])
        new_q = old_q + alpha * (reward + gamma * best_next_q - old_q)
        q_table[state, action] = new_q
        
        episode_reward += reward
        state = next_state
        path.append(state)
    
    # Decay exploration rate
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    episode_rewards.append(episode_reward)
    
    # Checkpoint: Pause and visualize
    if episode + 1 in checkpoints:
        print(f'\n✓ Checkpoint at Episode {episode + 1}')
        print(f'  Reward: {episode_reward}')
        print(f'  Epsilon (exploration): {epsilon:.3f}')
        print(f'  Recent avg reward: {np.mean(episode_rewards[-20:]):.1f}')
        
        checkpoint_data[episode + 1] = path
        
        # Show the path
        fig = draw_grid(path[-1], episode_num=episode + 1, title=f'Final position (Path length: {len(path)})')
        plt.show()

print('\n' + '='*60)
print('✅ TRAINING COMPLETE!')
print('='*60)
print(f'Final epsilon: {epsilon:.4f} (mostly exploiting, barely exploring)')
print(f'Final reward: {episode_rewards[-1]}')
print(f'Average last 50 episodes: {np.mean(episode_rewards[-50:]):.1f}')

## Cell 6: Analyze the Learning

Let's compare how the agent performed at different training stages:

In [ ]:
print('\n' + '='*60)
print('📊 LEARNING ANALYSIS')
print('='*60)

early_avg = np.mean(episode_rewards[:50])
mid_avg = np.mean(episode_rewards[200:250])
late_avg = np.mean(episode_rewards[-50:])

print(f'\n🟡 Early Training (Episodes 1-50):')
print(f'   Average Reward: {early_avg:.1f}')
print(f'   Status: Agent learning basics, falling off cliff often')

print(f'\n🟠 Mid Training (Episodes 200-250):')
print(f'   Average Reward: {mid_avg:.1f}')
print(f'   Status: Agent improving, finding better paths')

print(f'\n🟢 Late Training (Episodes 450-500):')
print(f'   Average Reward: {late_avg:.1f}')
print(f'   Status: Agent is EXPERT, optimal paths found!')

print(f'\n📈 Total Improvement: {late_avg - early_avg:.1f} reward points')
print(f'   (More negative = more cliff falls; 0 or positive = good paths)')

## Cell 7: Generate GIF Videos from Checkpoints

Take the checkpoint episodes and create animated GIFs showing the agent moving step-by-step.

In [ ]:
print('\n' + '='*60)
print('🎬 GENERATING ANIMATED GIFs')
print('='*60)

output_dir = Path('rl_training_videos')
output_dir.mkdir(exist_ok=True)

def create_grid_image(state, episode_num, title='', step_num=None):
    """Create PIL Image of the cliff grid"""
    img_width = 1200
    img_height = 300
    cell_width = img_width // 12
    cell_height = img_height // 4
    
    img = Image.new('RGB', (img_width, img_height), 'black')
    d = ImageDraw.Draw(img)
    
    # Draw grid
    for row in range(4):
        for col in range(12):
            x1 = col * cell_width
            y1 = row * cell_height
            x2 = x1 + cell_width
            y2 = y1 + cell_height
            
            # Determine cell color
            if row == 3 and col == 0:  # Start
                color = (0, 100, 0)  # Dark green
            elif row == 3 and col == 11:  # Goal
                color = (200, 150, 0)  # Gold
            elif row == 3 and 1 <= col <= 10:  # Cliff
                color = (150, 0, 0)  # Dark red
            else:
                color = (40, 40, 40)  # Dark gray
            
            d.rectangle([x1, y1, x2, y2], fill=color, outline='white')
    
    # Draw agent
    agent_row = state // 12
    agent_col = state % 12
    center_x = agent_col * cell_width + cell_width // 2
    center_y = agent_row * cell_height + cell_height // 2
    radius = min(cell_width, cell_height) // 3
    d.ellipse([center_x - radius, center_y - radius, center_x + radius, center_y + radius], 
              fill=(100, 150, 255), outline='white')
    
    # Add text
    d.text((10, 10), f'Episode {episode_num}', fill='white')
    if step_num is not None:
        d.text((10, 30), f'Step {step_num}', fill='cyan')
    
    return img

print('\nCreating GIFs for each checkpoint...')
for checkpoint_ep, path in checkpoint_data.items():
    print(f'  Episode {checkpoint_ep}: {len(path)} steps')
    
    frames = []
    for step_idx, state in enumerate(path):
        img = create_grid_image(state, checkpoint_ep, step_num=step_idx)
        frames.append(img)
    
    # Add final frame multiple times to pause on it
    frames.extend([frames[-1]] * 10)
    
    # Save GIF
    gif_name = f'episode_{checkpoint_ep:03d}.gif'
    output_path = output_dir / gif_name
    frames[0].save(output_path, save_all=True, append_images=frames[1:], 
                   duration=100, loop=0)
    
    file_size_kb = output_path.stat().st_size / 1024
    print(f'    ✓ Saved {gif_name} ({file_size_kb:.0f} KB)')

print('\n✅ All GIFs created!')

## Cell 8: Plot Learning Curves

Visualize how the agent's performance improved over training.

In [ ]:
print('\n' + '='*60)
print('📈 PLOTTING LEARNING PROGRESS')
print('='*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot 1: Raw rewards
ax1 = axes[0]
ax1.plot(episode_rewards, alpha=0.3, color='blue', label='Per episode')

# Moving average
window = 50
moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(episode_rewards)), moving_avg, linewidth=2.5, 
         color='darkblue', label=f'Moving avg ({window} eps)')

# Reference lines
ax1.axhline(y=-100, color='red', linestyle='--', alpha=0.5, label='Cliff fall (-100)')
ax1.axhline(y=-1, color='orange', linestyle='--', alpha=0.5, label='Optimal path (~-1)')

# Mark checkpoints
for checkpoint in checkpoints:
    if checkpoint in episode_rewards:
        idx = checkpoint - 1
        ax1.plot(idx, episode_rewards[idx], 'ro', markersize=8)

ax1.set_xlabel('Episode')
ax1.set_ylabel('Reward per Episode')
ax1.set_title('Learning Curve: Agent Improvement Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Comparison of early vs late
ax2 = axes[1]
phases = ['Early\n(1-50)', 'Mid\n(200-250)', 'Late\n(450-500)']
rewards_by_phase = [
    np.mean(episode_rewards[:50]),
    np.mean(episode_rewards[200:250]),
    np.mean(episode_rewards[-50:])
]
colors_by_phase = ['red', 'orange', 'green']

bars = ax2.bar(phases, rewards_by_phase, color=colors_by_phase, alpha=0.7, edgecolor='black')
ax2.axhline(y=-100, color='red', linestyle='--', alpha=0.3, label='Cliff penalty')
ax2.axhline(y=-1, color='green', linestyle='--', alpha=0.3, label='Optimal')

# Add value labels on bars
for bar, val in zip(bars, rewards_by_phase):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:.1f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

ax2.set_ylabel('Average Reward')
ax2.set_title('Training Phases Comparison')
ax2.set_ylim([-110, 0])
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'learning_analysis.png', dpi=100, bbox_inches='tight')
print('\n✓ Saved learning_analysis.png')
plt.show()

print('\n✅ COMPLETE! All visualizations ready!')

## Summary

### What We Learned:

✅ **Episode 1:** Agent stumbles randomly, falls off cliff immediately (-100)

✅ **Episode 20:** Agent learned to avoid cliff, takes long safe path (-25 to -30)

✅ **Episode 100:** Agent improving, finding shorter paths (-5 to -10)

✅ **Episode 300:** Agent getting risky, walking cliff edge (-3 to -2)

✅ **Episode 499:** Agent EXPERT, near-optimal speedrun on cliff (-1 or 0)

### The Q-Learning Process:

1. **Exploration:** Early episodes, agent takes random actions
2. **Learning:** Agent observes rewards and updates Q-Table
3. **Exploitation:** Later episodes, agent uses learned knowledge
4. **Convergence:** Final policy is near-optimal!

### Check Out the GIFs:

Open the files in `rl_training_videos/` to see:
- `episode_001.gif` - Total chaos, immediate cliff fall
- `episode_020.gif` - Cautious, long path
- `episode_100.gif` - Learning, reasonable path
- `episode_300.gif` - Risky, cliff edge
- `episode_499.gif` - Expert, optimal speedrun!